# 실습 2: 퍼셉트론을 쌓아 신경망 만들기

## 오늘 할 일 — 75분

이론 2장에서 계산한 **회귀 퍼셉트론 → 활성화 함수 → XOR 다층 퍼셉트론**을 같은 입력과 가중치로 확인한다.
마지막에는 이후 학습 실습에서 사용할 `nn.Sequential`로 신경망을 조립한다.
오늘은 **정해진 가중치로 계산**한다. 가중치를 학습하는 방법은 4주차 실습 3에서 다룬다.

**대응 이론:** [Ch02 퍼셉트론과 다층 퍼셉트론](../chapters/ch02.qmd). 실습 1의 텐서 생성·슬라이싱·속성·메서드를 사용한다.

- Google Colab 기본 CPU 런타임을 사용한다. 학습 데이터를 내려받거나 GPU를 설정할 필요는 없다.
- 각 구간의 시간에는 **설명·따라 하기·직접 해보기**가 포함되어 있다. 실습은 제출하거나 채점하지 않는다.
- 직접 해보기에서는 먼저 출력을 예상하고, 작성 셀의 `None`을 바꾼다. 막히면 힌트를 본다.
- 작성 후에는 결과를 출력하고 아래의 예상 결과와 비교한다.
- 해설은 문서 끝에 있다. 해설을 확인한 뒤에도 작성 셀로 돌아가 직접 실행한다.

| 시간 | 내용 |
|---|---|
| 0–5분 | 이론의 가중합을 행렬 연산으로 계산 |
| 5–25분 | 같은 계산을 `nn.Linear`로 옮기고 편향 변경 |
| 25–37분 | 같은 가중합에 활성화 적용 |
| 37–55분 | 이론의 XOR을 층별로 계산·비교 |
| 55–65분 | `nn.Sequential`과 파라미터 수 |
| 65–75분 | 종합 연습·정리 |

## 1. 이론의 가중합을 행렬로 계산하기 — 5분

이론의 첫 회귀 예제는 $z=0.5x_1+0.5x_2-0.3$이다.
아래 그림에서 **가중치를 곱해 합하는 부분**을 먼저 계산한다.

![이론 2장의 회귀 퍼셉트론: 입력, 가중치, 합, 활성화, 출력](https://ralbu85.github.io/lecture_deeplearning/assets/img/perceptron_regression.svg)

`X`의 한 행은 입력 한 건이다. 가중치 `W`는 **출력별로 한 행**, 편향 `b`는 **출력별로 하나**씩 담는다.
이는 뒤에서 사용할 PyTorch 층의 저장 방식과 같다.
`.T`는 행렬의 행과 열을 바꾼다. `W.T`로 두 가중치를 세로로 놓으면 실습 1의 행렬 곱을 적용할 수 있다.

In [ ]:
import torch

X = torch.tensor([[0., 0.],
                  [1., 0.],
                  [0., 1.],
                  [1., 1.]])
W = torch.tensor([[0.5, 0.5]])
b = torch.tensor([-0.3])

z_matrix = X @ W.T + b
print(z_matrix)

출력은 $(-0.3,0.2,0.2,0.7)$이다.
두 번째 입력 $(1,0)$을 손으로 계산하면 $0.5\times1+0.5\times0-0.3=0.2$로 같다.
회귀 예제의 활성화는 항등 함수 $g(z)=z$이므로 이 가중합이 그대로 예측값이다.

## 2. 같은 계산을 `nn.Linear`로 옮기기 — 20분

### `nn.Linear(입력 개수, 출력 개수)`

`torch.nn`은 신경망 기능을 모은 모듈이며, `as nn`은 짧게 부를 별명을 정한다.
`nn.Linear`는 입력에 가중치를 곱해 더하는 층을 만든다. 괄호 안의 두 숫자는 항상 다음 순서다.

```text
nn.Linear(입력 개수, 출력 개수)
```

**`nn.Linear(2, 1)`은 입력 두 개를 받아 출력 하나를 만드는 층이다.**
이 예제에서는 입력이 $x_1,x_2$ 두 개이고, 출력이 가중합 $z$ 하나다.

```text
x₁, x₂  →  nn.Linear(2, 1)  →  z
입력 2개                       출력 1개
```

출력 하나를 계산하려면 두 입력에 곱할 **가중치 2개**와 더할 **편향 1개**가 필요하다.
`layer = nn.Linear(2, 1)`로 층을 만들고, `layer(X)`로 입력을 넣어 계산한다.

In [ ]:
import torch.nn as nn

layer = nn.Linear(2, 1)   # 입력 2개 → 출력 1개

입력 4건을 넣으면 각 건에서 값 하나씩, 총 4개의 값이 나온다.
**입력 개수 2는 한 건에 있는 변수 수**다. 데이터가 4건이라고 `nn.Linear(4, 1)`로 쓰지 않는다.

층을 만들 때 가중치와 편향은 무작위로 초기화된다.
`weight`와 `bias` 속성으로 값을 읽을 수 있다.

In [ ]:
print(layer.weight)   # 출력 하나에 연결된 가중치 두 개
print(layer.bias)     # 출력 하나의 편향

### 앞에서 쓴 가중치를 그대로 넣는다 — 제공 코드

`copy_(텐서)`는 기존 파라미터에 값을 복사한다.
`with torch.no_grad():` 아래에서는 기울기를 기록하지 않고 값을 설정한다.
아래 두 줄로 앞에서 만든 `W`, `b`를 넣는다.

In [ ]:
with torch.no_grad():
    layer.weight.copy_(W)
    layer.bias.copy_(b)

z = layer(X)
print(z)

`layer(X)`는 입력 4건 각각에 대해 값 하나를 계산한다. 결과는 `(4, 1)`이다.
앞의 행렬 계산과 나란히 출력해 비교한다. 두 결과 모두 $(-0.3,0.2,0.2,0.7)$이다.

In [ ]:
print(z_matrix)
print(z)

| 직접 계산 | PyTorch 층 |
|---|---|
| `W` | `layer.weight` |
| `b` | `layer.bias` |
| `X @ W.T + b` | `layer(X)` |

이후에는 가중합을 `layer(X)`로 계산하고, 활성화 함수를 별도로 붙인다.

### 직접 해보기 — 생성과 호출 구분

입력 2개로 출력 1개를 만드는 새 층 `practice_layer`를 생성하고, 네 입력 `X`의 출력 `practice_z`를 구한다.

**작성 셀**

In [ ]:
# ✏️ 직접 채워 보세요
practice_layer = None
practice_z = None

힌트: 첫 줄에서는 층을 만들고, 다음 줄에서는 그 층으로 계산한다.

**출력 확인 — 작성 후 실행**

In [ ]:
# ✏️ 직접 채워 보세요
print(practice_z)
print(practice_z.shape)

출력은 4행 1열, `torch.Size([4, 1])`이다. 각 행은 입력 한 건의 출력이다.

**설명하기:** 층을 만드는 줄과 입력으로 계산하는 줄을 각각 짚어 본다.

### 편향만 바꾸면?

비교할 층을 하나 더 만든다. 가중치는 같고 편향만 -0.3에서 0.2로 바꾼다.

In [ ]:
shifted_layer = nn.Linear(2, 1)
with torch.no_grad():
    shifted_layer.weight.copy_(torch.tensor([[0.5, 0.5]]))
    shifted_layer.bias.copy_(torch.tensor([0.2]))

### 직접 해보기 — 편향의 효과 예상하기

새 층의 출력 네 값을 먼저 예상해서 `expected_shift`에 텐서로 적는다. 그다음 `shifted_layer(X)`를 `shifted_z`에 담는다.

**작성 셀**

In [ ]:
# ✏️ 직접 채워 보세요
expected_shift = None
shifted_z = None

힌트: 편향이 얼마나 증가했는지 먼저 계산한다.

**출력 확인 — 작성 후 실행**

In [ ]:
# ✏️ 직접 채워 보세요
print('예상:', expected_shift)
print('계산:', shifted_z)
print('변화량:', shifted_z - z)

계산 결과는 `(0.2, 0.7, 0.7, 1.2)`이다. 네 값 모두 원래보다 0.5씩 증가한다.

**설명하기:** 네 출력의 변화량이 같은 이유는 무엇인가?

## 3. 같은 합에 다른 활성화 적용 — 12분

이론은 **합을 계산하기 → 활성화에 통과시키기**의 두 단계다.
원래 층의 가중합 `z`를 그대로 두고 두 번째 단계만 바꾼다.

| 활성화 | 하는 일 | 이번에 쓰는 코드 |
|---|---|---|
| 항등 함수 | 합을 그대로 출력 | `z` |
| 계단 함수 | 0 이상이면 1, 아니면 0 | `step(z)` |
| Sigmoid | 합을 0과 1 사이의 값으로 변환 | `sigmoid = nn.Sigmoid()` 후 `sigmoid(z)` |
| ReLU | 음수만 0으로 변환 | `relu = nn.ReLU()` 후 `relu(z)` |

### 계단 함수를 정의한다

입력 텐서의 각 값이 0 이상이면 1, 아니면 0을 반환하는 함수다.
`.float()`는 비교 결과의 참·거짓을 실수 1과 0으로 바꾼다.

In [ ]:
def step(x):
    return (x >= 0).float()

step_pred = step(z)
print(step_pred)

결과는 `(0,1,1,1)`, 이론의 OR 출력이다. 이후에도 계단 활성화는 `step(...)`으로 적용한다.

### Sigmoid와 ReLU는 부품으로 적용한다

Sigmoid와 ReLU도 **부품 생성 → 입력을 넣어 호출** 순서로 사용한다. 두 부품에는 학습할 가중치나 편향이 없다.

In [ ]:
sigmoid = nn.Sigmoid()
relu = nn.ReLU()

Sigmoid에서는 마지막 입력 `(1,1)`의 합 0.7이 약 0.668로 바뀐다.

### 직접 해보기 — 활성화만 바꾸기

`sigmoid(z)`를 `prob`, `relu(z)`를 `relu_z`에 담는다. ReLU가 바꿀 행을 먼저 짚고, 실행 후 계단 함수 출력과 비교한다.

**작성 셀**

In [ ]:
# ✏️ 직접 채워 보세요
prob = None
relu_z = None

힌트: 부품은 이미 위에서 만들었다. 이번에는 입력을 넣어 호출한다.

**출력 확인 — 작성 후 실행**

In [ ]:
# ✏️ 직접 채워 보세요
print('Sigmoid:', prob)
print('ReLU:', relu_z)

Sigmoid는 약 `(0.426, 0.550, 0.550, 0.668)`, ReLU는 `(0, 0.2, 0.2, 0.7)`이다.

**설명하기:** 활성화를 바꾸었을 때 가중치도 바뀌었는가? 출력의 모양과 값 중 무엇이 바뀌었는가?

## 4. 이론의 XOR을 층별로 계산 — 18분

이론에서는 OR과 NAND를 계산하는 두 퍼셉트론의 출력을, AND를 계산하는 다음 퍼셉트론에 넣었다.
이번에도 **동일한 네 입력 `X`**를 사용한다.

```text
입력 x₁, x₂ → 가중합 두 개 → 계단 함수 → s₁, s₂ → 가중합 하나 → 계단 함수 → ŷ
               hidden_layer                         output_layer
```

| 노드 | 가중치 | 편향 | 활성화 |
|---|---|---|---|
| 은닉 $s_1$: OR | 0.5, 0.5 | -0.3 | 계단 |
| 은닉 $s_2$: NAND | -0.5, -0.5 | 0.7 | 계단 |
| 출력: AND | 0.5, 0.5 | -0.7 | 계단 |

### 여기서는 왜 `nn.Linear(2, 2)`인가?

앞의 `nn.Linear(2, 1)`은 두 입력으로 가중합 **하나**를 만들었다.
XOR의 은닉층에서는 **같은 $x_1,x_2$를 받는 퍼셉트론 두 개**가 필요하다. 따라서 입력은 여전히 2개이고 출력은 2개다.

```text
                         ┌→ z₁ =  0.5x₁ + 0.5x₂ − 0.3
x₁, x₂ → Linear(2, 2) ──┤
                         └→ z₂ = −0.5x₁ − 0.5x₂ + 0.7
입력 2개                    출력 2개
```

각 출력은 **두 입력을 모두 사용**하며, 자기 가중치와 편향을 가진다.
`weight`의 첫 행은 $z_1$의 가중치 두 개, 둘째 행은 $z_2$의 가중치 두 개다.
계단 함수를 적용하면 이 두 값이 은닉 출력 $s_1,s_2$가 된다.

출력층은 $s_1,s_2$ **두 개를 받아 가중합 하나**를 만들므로 `nn.Linear(2, 1)`이다.

| 층 | 받는 값 | 만드는 값 | 코드 |
|---|---|---|---|
| 은닉층 | $x_1,x_2$ 두 개 | $z_1,z_2$ 두 개 | `nn.Linear(2, 2)` |
| 출력층 | $s_1,s_2$ 두 개 | 가중합 하나 | `nn.Linear(2, 1)` |

**앞 층이 내보내는 개수와 다음 층이 받는 개수가 같아야 연결된다.**
여기서는 은닉 출력 2개를 출력층이 그대로 받는다. 아래는 이론의 가중치를 넣는 제공 코드다.

In [ ]:
hidden_layer = nn.Linear(2, 2)   # x₁, x₂ 두 개 → 가중합 두 개
output_layer = nn.Linear(2, 1)   # s₁, s₂ 두 개 → 가중합 하나

with torch.no_grad():
    hidden_layer.weight.copy_(torch.tensor([[0.5, 0.5], [-0.5, -0.5]]))
    hidden_layer.bias.copy_(torch.tensor([-0.3, 0.7]))
    output_layer.weight.copy_(torch.tensor([[0.5, 0.5]]))
    output_layer.bias.copy_(torch.tensor([-0.7]))

### 은닉층 출력이 다음 층의 입력이다

`hidden_z`는 가중합 두 개, `hidden_output`은 계단 함수를 적용한 두 값이다.
먼저 `(1,0)`인 두 번째 입력이 은닉층에서 어떤 두 값을 만드는지 계산한다.

In [ ]:
hidden_z = hidden_layer(X)
hidden_output = step(hidden_z)
print(hidden_z)
print(hidden_output)
print(hidden_output.shape)    # (4, 2): 데이터 4건, 은닉 출력 2개

### 직접 해보기 — 은닉 출력으로 XOR 완성하기

`hidden_output`을 출력층에 넣어 `output_z`를 구하고, `step`을 적용해 `xor_pred`를 만든다.

**작성 셀**

In [ ]:
# ✏️ 직접 채워 보세요
output_z = None
xor_pred = None

힌트: `output_layer(앞 층의 출력)`으로 가중합을 구한 뒤 `step(가중합)`을 계산한다.

**출력 확인 — 작성 후 실행**

In [ ]:
# ✏️ 직접 채워 보세요
print('출력층 가중합:', output_z)
print('최종 판정:', xor_pred)

가중합은 `(-0.2, 0.3, 0.3, -0.2)`, 최종 판정은 XOR 정답 `(0, 1, 1, 0)`이다.

**설명하기:** 단일 OR 퍼셉트론과 비교하면 어느 입력의 최종 판정이 달라졌는가?

### 은닉 활성화를 빼면 어떻게 될까?

가중치는 그대로 두고, 은닉 가중합을 계단 함수 없이 출력층에 바로 넣는다.
결과를 예상한 뒤 실행한다. 앞 문제를 아직 못 풀었어도 아래 비교는 실행할 수 있다.

In [ ]:
with_step_z = output_layer(hidden_output)
with_hidden_step = step(with_step_z)

without_step_z = output_layer(hidden_z)
without_hidden_step = step(without_step_z)
print('은닉 계단 있음:', with_hidden_step)
print('은닉 계단 없음:', without_hidden_step)

은닉 가중합 두 개의 합은 모든 행에서 0.4다. 따라서 은닉 활성화를 빼면 출력층의 합은 항상
$0.5\times0.4-0.7=-0.5$여서 네 판정이 모두 0이 된다.
이 비교는 **같은 가중치에서 은닉 활성화가 한 역할**을 보여준다.
활성화 없는 가중합 층들을 하나로 합칠 수 있다는 일반적인 이유는 이론의 해당 절과 연결한다.

계단 함수는 이론의 XOR 계산을 재현하기 위해 썼다. 뒤의 학습 실습에서는 ReLU나 Sigmoid 같은 활성화를 사용한다.

## 5. 자주 쓸 구조를 `nn.Sequential`로 묶기 — 10분

`nn.Sequential`은 전달한 부품을 **적힌 순서대로 실행하는 모형**이다.
각 부품의 출력이 다음 부품의 입력이 된다. 반환된 모형도 `model(X)`로 호출한다.

먼저 앞에서 만든 단일 층과 Sigmoid를 묶는다. **같은 부품을 묶으므로 가중치도 그대로**다.

In [ ]:
prob_model = nn.Sequential(layer, sigmoid)
print(prob_model(X))          # sigmoid(layer(X))와 같은 네 값

### 은닉층에 ReLU를 쓰는 회귀 모형

이제 이후 학습에서 쓸 형태를 만든다. 각 `Linear`를 새로 생성하므로 가중치도 새로 초기화된다.
이 모형은 출력 실수 하나를 예측할 구조이며, 앞의 XOR 정답을 재현하도록 설정한 모형이 아니다.

```text
입력 2개 → Linear(2, 2) → ReLU → Linear(2, 1) → 실수 출력 1개
```

회귀 출력은 항등 함수이므로 마지막 `Linear`의 결과를 그대로 쓴다.
중간 활성화는 `Sequential` 안에 명시적으로 넣어야 한다.

In [ ]:
model = nn.Sequential(
    nn.Linear(2, 2),
    nn.ReLU(),
    nn.Linear(2, 1),
)
print(model(X))

### 파라미터 수를 코드로 확인

이론에서 한 층의 파라미터 수는 **입력 수 × 출력 수 + 출력 수**다.
위 모형은 $(2\times2+2)+(2\times1+1)=9$개다.

`model.parameters()`는 모형의 가중치·편향 텐서들을 차례로 꺼낼 수 있게 한다.
`for parameter in ...:`는 텐서를 하나씩 꺼내 들여쓴 코드를 반복한다.
`parameter.numel()`은 그 텐서의 전체 원소 수를 반환한다.

In [ ]:
count = 0
for parameter in model.parameters():
    count = count + parameter.numel()
print(count)

**확인 질문:** ReLU에는 파라미터가 없는데, 모형에 넣는 이유는 무엇인가?

## 6. 종합 연습 — 10분

같은 네 입력으로 사용할 **입력 2 → 은닉 3(ReLU) → 출력 1(항등)** 회귀 모형을 만든다.
앞의 2→2→1 구조에서 은닉노드만 하나 늘어난 것이다.
첫 층은 입력 2개에서 은닉 값 3개를 만들고, 다음 층은 그 3개를 받아 출력 1개를 만든다.
어느 두 숫자가 서로 같아야 연결되는지 먼저 짚어 본다.

### 직접 해보기 — 그림을 읽고 모형 조립하기

1. 세 부품을 순서대로 넣어 `my_model`을 만든다.
2. 입력 `X`의 예측을 `my_pred`에 담는다. 첫 입력 한 건은 `X[:1]`로 골라 예측을 `my_one`에 담는다.
3. 파라미터 수를 먼저 계산해 `expected_count`에 적는다.

**작성 셀**

In [ ]:
# ✏️ 직접 채워 보세요
my_model = None
my_pred = None
my_one = None
expected_count = None

힌트: 은닉노드 수가 바뀌면 앞 층의 출력 수와 뒤 층의 입력 수를 함께 바꾼다.

**출력 확인 — 작성 후 실행**

In [ ]:
# ✏️ 직접 채워 보세요
print(my_model)
print(my_pred)
print(my_pred.shape)
print(my_one)
print(expected_count)

모형은 `Linear(2, 3) → ReLU → Linear(3, 1)`이다. 네 건의 출력은 4행 1열이고, `my_one`은 그 첫 행과 같다. 파라미터 수는 13개다.

**설명하기:** 출력 모양이 맞다는 것만으로 잘 예측하는 모형이라고 할 수 있는가? 먼저 끝났다면 은닉노드를 4개로 바꾸고 증가한 파라미터 수를 설명한다.

## 마무리

이론의 계산은 **가중합 → 활성화**, 다층 계산은 **앞 층의 출력 → 다음 층의 입력**이다.
코드에서는 **부품 생성 → 호출 → 순서대로 연결**로 구현했다.
3주차에는 손실 계산을 손으로 연습하고, **4주차** [실습 3](lab03.qmd)에서는 가중치를 학습한다.

---

## 해설 — 먼저 직접 풀고 확인하기

해설 코드를 해당 작성 셀에 옮긴 후 결과를 출력해 비교한다.

### 생성과 호출 구분

```python
practice_layer = nn.Linear(2, 1)
practice_z = practice_layer(X)
```

### 편향의 효과 예상하기

```python
expected_shift = torch.tensor([[0.2], [0.7], [0.7], [1.2]])
shifted_z = shifted_layer(X)
```

### 활성화만 바꾸기

```python
prob = sigmoid(z)
relu_z = relu(z)
```

### 은닉 출력으로 XOR 완성하기

```python
output_z = output_layer(hidden_output)
xor_pred = step(output_z)
```

### 그림을 읽고 모형 조립하기

```python
my_model = nn.Sequential(
    nn.Linear(2, 3),
    nn.ReLU(),
    nn.Linear(3, 1),
)
my_pred = my_model(X)
my_one = my_model(X[:1])
expected_count = (2 * 3 + 3) + (3 * 1 + 1)
```

추가 연습 해석: 은닉노드가 4개면 파라미터 수는 $(2×4+4)+(4×1+1)=17$개다. 3개일 때보다 4개 늘어난다.
XOR 비교에서는 단일 OR이 1로 판정했던 `(1,1)`을 다층 모형이 0으로 판정한다.